In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
""",
    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
"""
}

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40, separators=["\n\n", "\n", ". ", " ", ""])

client = chromadb.Client()
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.create_collection(name="rag_chunks", embedding_function=hf_embedder)

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    collection.add(
        documents=doc_chunks,
        metadatas=[{"source": source, "chunk_index": i} for i in range(len(doc_chunks))],
        ids=[f"{source}_chunk_{i}" for i in range(len(doc_chunks))]
    )

print("Total chunks stored:", collection.count())

In [ ]:
# Add a third document to the collection

third_document = {
    "fitness_tracking.txt": """Fitness Tracking with Wearable Devices

Smartwatches and fitness bands can collect information such as
steps, heart rate, calories burned, sleep duration, and workout data.

These measurements can be analyzed to provide personalized fitness
insights and help users understand their daily activity patterns."""
}

for source, text in third_document.items():
    doc_chunks = splitter.split_text(text)

    collection.add(
        documents=doc_chunks,
        metadatas=[
            {"source": source, "chunk_index": i}
            for i in range(len(doc_chunks))
        ],
        ids=[
            f"{source}_chunk_{i}"
            for i in range(len(doc_chunks))
        ]
    )

print("Total chunks after adding third document:", collection.count())


In [ ]:
user_query = "How does RAG improve answers from a language model?"

results = collection.query(
    query_texts=[user_query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

In [ ]:
def distance_to_similarity(distance):
    return 1 - distance

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    similarity = distance_to_similarity(distance)
    print(f"Similarity: {similarity:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")

In [ ]:
def retrieve_with_threshold(query, collection, n_results=5, similarity_threshold=0.3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    filtered = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = distance_to_similarity(distance)
        if similarity >= similarity_threshold:
            filtered.append({"text": doc, "metadata": meta, "similarity": similarity})

    return filtered

# A relevant query
relevant_matches = retrieve_with_threshold("What is deep learning?", collection, similarity_threshold=0.3)
print(f"Relevant query -> {len(relevant_matches)} chunk(s) passed the threshold\n")
for m in relevant_matches:
    print(f"Similarity: {m['similarity']:.4f} | {m['text'][:80]}...")

In [ ]:
# Lower the similarity threshold to 0.1 and re-run
# the "chocolate cake" query

unrelated_matches = retrieve_with_threshold(
    "What is the best recipe for chocolate cake?",
    collection,
    similarity_threshold=0.1
)

print(
    f"Chocolate cake query -> "
    f"{len(unrelated_matches)} chunk(s) passed the 0.1 threshold"
)

print(
    "A lower threshold allows more chunks to pass, "
    "including weaker or unrelated matches."
)

print(
    "This shows that a good threshold should balance "
    "relevant results and unwanted noise."
)

if len(unrelated_matches) == 0:
    print(
        "No relevant context found - a real RAG system "
        "should say 'I don't know' here instead of guessing."
    )

In [ ]:
results = collection.query(
    query_texts=["how does search work with meaning instead of keywords"],
    n_results=2,
    where={"source": "embeddings_in_practice.txt"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Similarity: {distance_to_similarity(distance):.4f} | Source: {meta['source']}")
    print(doc)
    print("-" * 60)

In [ ]:
def retrieve_context(query, collection, n_results=5, similarity_threshold=0.25, source_filter=None, max_chunks=2000):
    query_kwargs = {
        "query_texts": [query],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"]
    }
    if source_filter:
        query_kwargs["where"] = {"source": source_filter}

    results = collection.query(**query_kwargs)

    context_chunks = []
    total_chars = 0

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = distance_to_similarity(distance)

        if similarity >= similarity_threshold:
            # Keep the TOTAL combined character length under max_chunks
            if total_chars + len(doc) > max_chunks:
                break

            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })

            total_chars += len(doc)

    return context_chunks


context = retrieve_context(
    "What is machine learning?",
    collection,
    max_chunks=500
)

print("Retrieved chunks:", len(context))
print("Total combined characters:", sum(len(c["text"]) for c in context))

for c in context:
    print(c)


In [ ]:
# Confirm retrieve_context() can pull chunks from the third document

third_results = retrieve_context(
    "What information can smartwatches collect?",
    collection,
    n_results=3,
    similarity_threshold=0.1,
    source_filter="fitness_tracking.txt",
    max_chunks=1000
)

print("Third document results:", len(third_results))

for result in third_results:
    print(
        f"Similarity: {result['similarity']:.4f} | "
        f"Source: {result['source']} | "
        f"Chunk #{result['chunk_index']}"
    )
    print(result["text"])
    print("-" * 60)


In [ ]:
def format_context_for_llm(query, context_chunks):

    if not context_chunks:
        return (
            f"Question: {query}\n\n"
            "No relevant context was found in the knowledge base."
        )

    context_text = "\n\n".join(
        f"[Source: {c['source']}, "
        f"chunk {c['chunk_index']}]\n"
        f"{c['text']}"
        for c in context_chunks
    )

    prompt = f"""
Answer the question using ONLY the context below.

If the answer isn't in the context, say you don't know.

Context:

{context_text}

Question:
{query}

Answer:
"""

    return prompt


context = retrieve_context(
    "How does RAG combine search with a language model?",
    collection,
    max_chunks=1500
)

final_prompt = format_context_for_llm(
    "How does RAG combine search with a language model?",
    context
)

print(final_prompt)

In [ ]:
test_queries = [
    "Why do companies use vector databases?",
    "What's the difference between AI and machine learning?",
    "How do chatbots understand differently worded questions?"
]

for q in test_queries:

    print(f"Query: {q}")

    matches = retrieve_context(
        q,
        collection,
        n_results=2,
        max_chunks=1000
    )

    for m in matches:

        print(
            f"   [{m['similarity']}] "
            f"({m['source']}) "
            f"{m['text'][:90]}..."
        )

    print()